In [ ]:
import gzip
import torch
import matplotlib.pyplot as plt
import einops

import json
import re
from os.path import join as opj
import numpy as np
from ts2.data.transforms import SRHBaseTransform
from PIL import Image
from torchvision.transforms.functional import adjust_contrast, adjust_brightness


In [ ]:
!ls /nfs/turbo/umms-tocho-snr/exp/chengjia/ts2/fmi_hdinov2_tile/643515de_Apr24-13-21-52_sd1000_dev_tune0/models/eval/training_39999/a2d1e04d_Apr25-16-00-00_sd1000_smpt_AT_epoch0-step124999/predictions/val_predictions.pt.gz

In [ ]:
#with gzip.open("/nfs/turbo/umms-tocho-snr/exp/chengjia/ts2/fmi_hdinov2_tile/789ec8bc_Apr13-01-37-13_sd1000_dev_tune0/models/eval/training_124999/75d0d319_Apr15-23-27-43_sd1000_smpt_SE_epoch0-step124999/predictions/val_predictions.pt.gz", "r") as fd:
#    data = torch.load(fd)

with gzip.open("/nfs/turbo/umms-tocho-snr/exp/chengjia/ts2/fmi_hdinov2_tile/643515de_Apr24-13-21-52_sd1000_dev_tune0/models/eval/training_39999/a2d1e04d_Apr25-16-00-00_sd1000_smpt_AT_epoch0-step124999/predictions/val_predictions.pt.gz", "r") as fd:
     data = torch.load(fd) 
# 75d0d319_Apr15-23-27-43_sd1000_smpt_SE_epoch0-step124999
# 01e6f853_Apr16-13-57-43_sd1000_rmbg_SE_epoch0-step124999

In [ ]:

class SRHCellImageGetter():

    def __init__(self, dset_root: str, return_str: bool = True):
        self.dset_root = dset_root
        self.return_str = return_str
        self.sbt = SRHBaseTransform()

    @staticmethod
    def get_nio_num_components(image_str: str):
        pattern = r'(?P<accession>(?P<nioinv>NIO|INV)_(?P<instution>[A-Z]+)_(?P<id>\d+[a-z]*))-(?P<mosaic>[a-zA-Z0-9_]+)-.*@(?P<cell_id>\d+)'
        match = re.match(pattern, image_str)
        assert match

        return match.groupdict()

    @staticmethod
    def im_to_bytestr(im):
        image = Image.fromarray(
            (255 * im.permute(1, 2, 0).numpy()).astype(np.uint8))
        output = io.BytesIO()
        image.save(output, format='JPEG')
        return "data:image/jpeg;base64," + base64.b64encode(
            output.getvalue()).decode()

    def get_image(self, row):
        nionc = self.get_nio_num_components(row["path"])

        try:
            with open(opj(self.dset_root, f"{nionc['accession']}_{nionc['mosaic']}_meta.json")) as fd:
                tensor_shape = json.load(fd)["tensor_shape"]

            mm_path = opj(self.dset_root, f"{nionc['accession']}_{nionc['mosaic']}_cells.dat")
            fd = np.memmap(mm_path,
                            dtype="uint16",
                            mode="r",
                            shape=tuple(tensor_shape))
            im = torch.tensor(np.array(fd[int(nionc["cell_id"]), :2, ...]).astype(float)).contiguous()
            im = adjust_brightness(adjust_contrast(self.sbt(im), 2), 3)
            fd._mmap.close()
            del fd
        except:
            print(row["path"])
            print(nionc)
            im = torch.zeros(3,64,64)

        if self.return_str:
            return self.im_to_bytestr(im)
        else:
            return im

    def __call__(self, row):
        return self.get_image(row)



In [ ]:
from sklearn.decomposition import PCA
import matplotlib.backends.backend_pdf

In [ ]:
scig = SRHCellImageGetter(dset_root="/nfs/turbo/umms-tocho-snr/exp/chengjia/scsrh_repl_root2", return_str=False)

In [ ]:
attns_raw = torch.clone(data["attns"][:,:,:,0,1:])
attns = torch.clone(data["attns"][:,:,:,0,1:])

In [ ]:
attns -= attns.min(dim=3, keepdim=True)[0]
attns /= attns.max(dim=3, keepdim=True)[0]

In [ ]:
pdf = matplotlib.backends.backend_pdf.PdfPages("pfm_attn_scores.pdf")

In [ ]:
for i in range(100):
    fig, axs = plt.subplots(1,3, figsize=(20, 5), gridspec_kw={'width_ratios': [10, 3,3]})
    #fig, axs = plt.subplots(1,4, figsize=(20, 5), gridspec_kw={'width_ratios': [10, 10, 1,1]})
    im = scig.get_image({"path": data["path"][i]})
    
    pca = PCA(n_components=3)
    pca.fit(attns_raw[i,-1].T)
    pca_features = pca.transform(attns_raw[i,-1].T)
    pca_features2 = pca_features - pca_features.min()
    pca_features2 = pca_features2 / pca_features2.max()


    #axs[0].imshow(einops.rearrange(attns_raw[i], "l a (h w) -> (a h) (l w)", h=12))
    axs[0].imshow(einops.rearrange(attns[i], "l a (h w) -> (a h) (l w)", h=12))
    axs[1].imshow((255 * im.permute(1, 2, 0).numpy()).astype(np.uint8)[8:-8,8:-8,:])
    axs[2].imshow(einops.rearrange(pca_features2, "(h w) c -> h w c", h=12))
    
    
    axs[0].axis("off")
    axs[1].axis("off")
    axs[2].axis("off")
    #axs[3].axis("off")
    
    plt.tight_layout()
    pdf.savefig(fig)

pdf.close()

In [ ]:
glob("../train/sep_viz/sep_token_tv/*.pt")

In [ ]:
import torch
from glob import glob
from PIL import Image

In [ ]:
for fn in glob("../train/sep_viz/sep_mean_v/*.pt"):
    data = torch.load(fn, map_location="cpu")
    im = einops.rearrange(adjust_brightness(adjust_contrast(data["image"][90, 0, ...], 2),3), "c h w -> h w c").cpu()
    Image.fromarray((im.numpy() * 255).astype(np.uint8)).save(fn+".png")

In [ ]:
fn = glob("../train/real_perturb/*.pt")[0]
data = torch.load(fn, map_location="cpu")
im = einops.rearrange(adjust_brightness(adjust_contrast(data["image"][2, 0, ...], 2),3), "c h w -> h w c").cpu()
Image.fromarray((im.numpy() * 255).astype(np.uint8)).save(fn+".png")

In [ ]:
data.keys()

In [ ]:
data["image"].shape

In [ ]:
for i in range(1):
    plt.figure()
    plt.imshow(einops.rearrange(adjust_brightness(adjust_contrast(data["image"][i, 0, ...], 2),3), "c h w -> h w c").cpu())
    plt.title(f"{i}")

In [ ]:
plt.imshow(einops.rearrange(data["alt_fg"][10], "c h w -> h w c"))

In [ ]:
plt.imshow(einops.rearrange(data["bg"][1], "c h w -> h w c"))